# Taxi Upfront Pricing Precision — V2 Analysis

This notebook delivers an executive-ready analysis of metered vs upfront pricing precision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_PATH = "../data/raw_data/test.csv"

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# Keep completed rides and core price fields
base = df.loc[df["b_state"].eq("finished")].copy()
base = base.dropna(subset=["metered_price", "upfront_price"])
base = base.loc[base["upfront_price"] > 0].copy()

# Derived pricing precision fields
base["price_difference"] = base["metered_price"] - base["upfront_price"]
base["relative_error"] = (base["price_difference"].abs() / base["upfront_price"]).astype(float)
base["is_switched"] = (base["relative_error"] > 0.20).astype(int)

# Bias features for root-cause analysis
base["distance_bias"] = (base["distance"] - base["predicted_distance"]) / base["predicted_distance"].replace(0, np.nan)
base["duration_bias"] = (base["duration"] - base["predicted_duration"]) / base["predicted_duration"].replace(0, np.nan)

print("clean rides:", len(base))
print("switch rate:", base["is_switched"].mean())

In [ ]:
# Metric definition
# Pricing precision failure rate = % of rides where upfront error exceeds 20%
pricing_precision_failure_rate = base["is_switched"].mean()
pricing_precision_success_rate = 1 - pricing_precision_failure_rate

print(f"Precision success rate: {pricing_precision_success_rate:.2%}")
print(f"Switch rate (failure): {pricing_precision_failure_rate:.2%}")

In [ ]:
# Segment diagnostics
def segment_switch_table(frame, col, min_rides=30):
    out = (frame.groupby(col, dropna=False)
           .agg(rides=("order_id_new", "count"),
                switch_rate=("is_switched", "mean"),
                median_rel_error=("relative_error", "median"))
           .reset_index()
           .sort_values("switch_rate", ascending=False))
    return out.loc[out["rides"] >= min_rides]

for c in ["gps_confidence", "eu_indicator", "entered_by", "dest_change_number", "prediction_price_type", "fraud_score"]:
    if c in base.columns:
        print("\n", c)
        if c == "gps_confidence":
            tmp = base.assign(gps_bucket=pd.cut(base[c], bins=[-np.inf,0.4,0.6,0.8,np.inf], labels=["<0.4","0.4-0.6","0.6-0.8",">=0.8"]))
            print(segment_switch_table(tmp, "gps_bucket", min_rides=1))
        elif c == "fraud_score":
            tmp = base.assign(fraud_bucket=pd.cut(base[c], bins=[-np.inf,0.2,0.5,0.8,np.inf], labels=["low","mid","high","very_high"]))
            print(segment_switch_table(tmp, "fraud_bucket", min_rides=1))
        else:
            print(segment_switch_table(base, c))

In [ ]:
# Predicted vs actual drift
bias = base[["is_switched", "distance_bias", "duration_bias"]].copy()
summary = bias.groupby("is_switched").agg({"distance_bias":"mean", "duration_bias":"mean"})
summary.index = ["not_switched", "switched"]
summary

In [ ]:
# Visuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(base["relative_error"], bins=50, ax=axes[0], color="steelblue")
axes[0].axvline(0.2, color="red", linestyle="--", label="20% switch threshold")
axes[0].set_title("Relative Pricing Error Distribution")
axes[0].legend()

gps_plot = base.assign(gps_bucket=pd.cut(base["gps_confidence"], bins=[-np.inf,0.4,0.6,0.8,np.inf], labels=["<0.4","0.4-0.6","0.6-0.8",">=0.8"]))
(gps_plot.groupby("gps_bucket")["is_switched"].mean()
 .reindex(["<0.4","0.4-0.6","0.6-0.8",">=0.8"])
 .plot(kind="bar", ax=axes[1], color="darkorange"))
axes[1].set_title("Switch Rate by GPS Confidence")
axes[1].set_ylabel("Switch rate")

plt.tight_layout()
plt.show()

## Top opportunities (prioritized)
1. **GPS-quality fallback pricing** for low-confidence trips.
2. **Dynamic uplift for underpredicted duration/distance risk** (proxy via historical bias + live trip context).

These two opportunities combine large impact with straightforward rollout feasibility.